In [1]:
"""
train_model.py
Trains an AdaBoost REGRESSION model to predict the numeric US_AQI value
from 6 pollutant readings (PM2.5, PM10, NO2, SO2, O3, CO).

Dataset: INDIA_AQI_COMPLETE_20251126.csv (842,160 real hourly readings,
29 Indian cities).
"""

import pandas as pd
import joblib
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# ---- 1. Load only the columns we need (dataset has 60+ columns, 842k rows) ----
DATA_PATH = "INDIA_AQI_COMPLETE_20251126.csv"

FEATURES = ["PM2_5_ugm3", "PM10_ugm3", "NO2_ugm3", "SO2_ugm3", "O3_ugm3", "CO_ugm3"]
TARGET = "US_AQI"

usecols = FEATURES + [TARGET]
print("Loading dataset (this may take a moment for 842k rows)...")
df = pd.read_csv(DATA_PATH, usecols=usecols)

# ---- 2. Drop rows with missing pollutant or target values ----
before = len(df)
df = df.dropna()
print(f"Loaded {before} rows -> {len(df)} rows after dropping missing values")

X = df[FEATURES]
y = df[TARGET]

# ---- 3. Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- 4. Train AdaBoost Regressor ----
# Shallow trees + moderate learning rate work best here: the dataset has a
# small number of extreme hazardous-event outliers (AQI > 500) that can
# otherwise dominate the boosting loss.
base_estimator = DecisionTreeRegressor(max_depth=3)
model = AdaBoostRegressor(
    estimator=base_estimator,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42,
)

print("Training AdaBoost model...")
t0 = time.time()
model.fit(X_train, y_train)
print(f"Done in {time.time() - t0:.1f}s")

# ---- 5. Evaluate ----
y_pred = model.predict(X_test)
print("\n--- Evaluation on test set ---")
print("MAE :", round(mean_absolute_error(y_test, y_pred), 2))
print("RMSE:", round(root_mean_squared_error(y_test, y_pred), 2))
print("R2  :", round(r2_score(y_test, y_pred), 3))

# ---- 6. Save the trained model ----
joblib.dump(model, "aqi_regressor.joblib")
print("\nSaved: aqi_regressor.joblib")

Loading dataset (this may take a moment for 842k rows)...
Loaded 842160 rows -> 842015 rows after dropping missing values
Training AdaBoost model...
Done in 46.5s

--- Evaluation on test set ---
MAE : 21.8
RMSE: 42.82
R2  : 0.635

Saved: aqi_regressor.joblib
